# RoboBarista — Welches Kamera-/AprilTag-Setup ist am besten?

Wir vergleichen **1 vs. 2 seitliche AprilTags**, **3 vs. 4 Kameras** und **OpenCV vs. echtes AprilTag 3 (Pupil Labs)**. Insgesamt acht Kombinationen, jeweils mit den gleichen drei Testpositionen und Neigungen, 1280 × 960 Pixeln und einem festen Zufalls-Seed.

**Warum?** Unsere bisherigen 2–3 mm waren lediglich eine erste Baseline. Wir prüfen jetzt unabhängig die Fehler an der Ausgießerspitze, den Winkel und die Erkennungsrate.

**Genauigkeits-Ziel:** maximal 0,5 mm Spitzenfehler und 1° Winkel. Das ist ein Forschungsziel, nicht vorab erreicht. Unsere 3 Pose-Beispiele sind noch keine flüssige/dynamische Bewegung. Die Kamera- und Tag-Positionen sind zunächst Kandidaten, nicht wissenschaftlich optimiert.

**Kosten:** CPU ohne GPU möglich. Die Tests rendern mit OSMesa, wenn du in Colab Hardwarebeschleuniger „Keine“ auswählst.

Wähle **Laufzeit → Alle ausführen**. Die Ergebnisse werden in `runs/rig_comparison` gespeichert. Die letzte Zelle zeigt eine Vergleichstabelle und Beispielbilder.


In [ ]:
# 1. GitHub-Projekt laden (keine ZIP nötig)
import os, subprocess, sys, pathlib
repo = pathlib.Path('/content/robobarista-vision')
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/Coffee-Tobi/robocasa.git', str(repo)], check=True)
else:
    # Bei erneutem Ausführen aktuelle Verbesserungen von GitHub holen.
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
os.chdir(repo)
print('Projekt:', repo)


In [ ]:
# 2. Systemgrafik (CPU/OSMesa) und Python-Pakete installieren
# Ohne GPU benötigt MuJoCo echte OSMesa-Systembibliotheken; pip allein reicht nicht.
import os, subprocess, sys, ctypes.util
subprocess.run(['apt-get', '-qq', 'update'], check=True)
subprocess.run(['apt-get', '-qq', 'install', '-y', 'libosmesa6-dev',
                'libgl1', 'libglx-mesa0', 'libgl1-mesa-dri'], check=True)
osmesa_lib = ctypes.util.find_library('OSMesa')
print('OSMesa-Systembibliothek:', osmesa_lib)
if not osmesa_lib:
    raise RuntimeError('OSMesa fehlt trotz apt-Installation. Bitte die Ausgabe von Schritt 2 teilen.')

subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', '-r', 'requirements.txt'], check=True)

# Eigenständiger Importcheck im richtigen Rendering-Backend (ohne GPU: OSMesa).
backend = 'egl' if os.path.exists('/dev/nvidia0') else 'osmesa'
check_env = os.environ.copy()
check_env['MUJOCO_GL'] = backend
if backend == 'osmesa':
    check_env['PYOPENGL_PLATFORM'] = 'osmesa'
print('MuJoCo-Importtest mit Backend:', backend, flush=True)
subprocess.run([sys.executable, '-c',
                'import mujoco; print("MuJoCo:", mujoco.__version__, "Import OK")'],
               env=check_env, check=True)
import cv2, numpy as np
print('OpenCV:', cv2.__version__, 'NumPy:', np.__version__)


In [ ]:
# 3. Schnelle Tests der Kamera-Mathematik — unabhängig von MuJoCo
import subprocess,sys
subprocess.run([sys.executable,'-m','pytest','-q','tests'],check=True)


In [ ]:
# 4. Acht Varianten unter gleichen Bedingungen durchrechnen (kein GPU-Zwang)
import os,sys,subprocess
env=os.environ.copy()
env['MUJOCO_GL']='egl' if os.path.exists('/dev/nvidia0') else 'osmesa'
if env['MUJOCO_GL']=='osmesa':
    env['PYOPENGL_PLATFORM']='osmesa'
print("Rendering:",env['MUJOCO_GL'],flush=True)
cmd=[sys.executable,'-u','-m','robobarista.compare_rigs',
     '--trials','3','--width','1280','--height','960']
result=subprocess.run(cmd,cwd=str(repo),env=env,
    stdout=subprocess.PIPE,stderr=subprocess.PIPE,text=True,errors='replace')
print(result.stdout if result.stdout else '(Keine Standardausgabe)')
print('=== Fehlerhinweise ===')
print(result.stderr[-12000:] if result.stderr else '(keine)')
print('Gesamt-Exit-Code:',result.returncode)


In [ ]:
# 5. Vergleichstabelle, 4-Kamera-Montage und ZIP zum Sichern
from pathlib import Path
import json, pandas as pd
from IPython.display import display, Image, FileLink
import zipfile
folder=Path(repo)/'runs/rig_comparison'
p=folder/'comparison.json'
if not p.exists():
    print('Noch kein Bericht vorhanden. Bitte Ausgabe aus Schritt 4 schicken.')
else:
    rows=json.loads(p.read_text())
    frame=pd.DataFrame(rows)
    columns=['variant','status','pitcher_valid_trials','min_pitcher_cameras_seen',
             'max_spout_tip_mm','max_pitcher_angle_deg',
             'max_relative_spout_to_cup_mm','exit_code']
    display(frame[columns])
    print('PASS erfordert alle Messungen, mindestens 2 Kameras sowie 0,5 mm/1°.')
    print('Der relative Spitze-zu-Tasse-Fehler wird separat ausgewiesen.')
    for variant in ('pupil_1tag_4cam','pupil_2tag_4cam'):
        image_path=folder/variant/'rgbd_montage.jpg'
        if image_path.exists():
            print('Kameraübersicht:',variant)
            display(Image(filename=str(image_path)))
    archive=Path(repo)/'runs/rig_comparison_results.zip'
    with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for file in folder.rglob('*'):
            if file.is_file() and file.suffix!='.npy':
                z.write(file,file.relative_to(Path(repo)/'runs'))
    print('Messberichte und Bilder herunterladen:')
    display(FileLink(str(archive)))


### Interpretation

Die Marker werden auf starren virtuellen Trägern seitlich am Kännchen angebracht. Die drei Kameras sind nicht alle auf einer Seite: zwei zeigen auf die nahe Seite und eine auf die ferne Seite. Kamera 4 ist eine zusätzliche Nahansicht. Die Kameraanordnung bleibt zwischen allen vier mechanischen Varianten identisch, damit die Ergebnisse vergleichbar sind.

Das Benchmark-Ergebnis stammt aus einer vereinfachten virtuellen Szene mit ideal bekannter Kamerakalibrierung und vereinfachter Ausgießergeometrie. Wir benötigen vor einer realen Umsetzung noch Tests mit Bildrauschen, Verdeckungen, zeitlicher Bewegung und Montagefehlern. Nicht bestandene Ergebnisse sind kein Grund, Prüfgrenzen zu lockern.
